# VEGFR2 Activity Prediction - Colab Notebook

This notebook provides a complete workflow for:
1. **Environment setup** - Install dependencies
2. **Data download** - Fetch ChEMBL VEGFR2 IC50 data
3. **Model training** - Train ML (RF, SVM, XGBoost) and GNN (GCN, GAT, MPNN) models
4. **Screening/Inference** - Predict on new compound libraries

## Requirements
- GPU runtime (required for GNN training)
- ~10-15 min for full training

**Enable GPU**: Runtime → Change runtime type → GPU

## 1. Environment Setup

In [1]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. GNN training will be VERY slow on CPU.")
    print("   Enable GPU: Runtime → Change runtime type → GPU")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# Install dependencies

%pip install -q numpy pandas pyyaml scikit-learn xgboost rdkit optuna
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118   
# Verify imports
import torch, numpy, pandas, sklearn, xgboost, rdkit, yaml, optuna
print("✅ All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 61.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 29.7 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu118
✅ All packages installed


In [16]:
# Clone the repository
!rm -f /content/ai-code/data/processed/*.csv
!cd /content/ai-code && git pull
!cd /content/ai-code && pip install -e .
import os
REPO_URL = "https://github.com/Techbjd/ai-code.git"  # CHANGE THIS
REPO_DIR = "/content/ai-code"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already exists, pulling latest...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working in: {os.getcwd()}")

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.79 KiB | 612.00 KiB/s, done.
From https://github.com/Techbjd/ai-code
   a3de7c8..6f95d10  main       -> origin/main
Updating a3de7c8..6f95d10
Fast-forward
 scripts/train.py   |  10 ++--
 vegfr2_colab.ipynb | 141 ++++++++++++++++++++++++++++++++++++++++-------------
 2 files changed, 112 insertions(+), 39 deletions(-)
Obtaining file:///content/ai-code
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vegfr2 (pyproject.toml) ... done
  Created wheel for vegfr2: filename=vegfr2-0.1.0-0.editable-py3-none-any.whl size=2868 sha256=d10abaf046f3d98e19faa2491b3293b383d71e4f8c0cf3751e1bfde

In [4]:
# Install the package in development mode
%pip install git+https://github.com/Techbjd/ai-code.git -q

# Verify import works
from vegfr2.features import mol_to_graph, ATOM_FEAT_DIM, BOND_FEAT_DIM
from vegfr2.gnn_models import build_model
from vegfr2.ml_models import train_ml_model
print(f"✅ Package imported successfully")
print(f"   Atom feat dim: {ATOM_FEAT_DIM}")
print(f"   Bond feat dim: {BOND_FEAT_DIM}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Package imported successfully
   Atom feat dim: 32
   Bond feat dim: 11


## 2. Download Training Data

In [18]:
# Download pre-processed train/val/test splits from GitHub
!mkdir -p data/processed
!wget -q -O data/processed/train.csv https://raw.githubusercontent.com/Techbjd/ai-code/main/data/processed/train.csv
!wget -q -O data/processed/val.csv https://raw.githubusercontent.com/Techbjd/ai-code/main/data/processed/val.csv
!wget -q -O data/processed/test.csv https://raw.githubusercontent.com/Techbjd/ai-code/main/data/processed/test.csv

# Verify data
import pandas as pd
train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")
test_df = pd.read_csv("data/processed/test.csv")
print(f"Train: {len(train_df)} (active: {train_df['active'].sum()}, inactive: {(train_df['active']==0).sum()})")
print(f"Val: {len(val_df)} (active: {val_df['active'].sum()}, inactive: {(val_df['active']==0).sum()})")
print(f"Test: {len(test_df)} (active: {test_df['active'].sum()}, inactive: {(test_df['active']==0).sum()})")

Train: 7834 (active: 4441, inactive: 3393)
Val: 980 (active: 556, inactive: 424)
Test: 980 (active: 556, inactive: 424)


## 3. Train Models

Choose which models to train. Each cell trains one model type.

In [ ]:
# Train Classical ML models (RF, SVM, XGBoost) using pre-processed splits
# Fast - runs on CPU, ~1-2 minutes
!python scripts/train.py --model rf --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
!python scripts/train.py --model svm --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
!python scripts/train.py --model xgb --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv

# Results saved to runs/<model>/model.pkl
# Metrics saved to runs/results.json

Loaded pre-processed data: train=7834 val=980 test=980

=== Training RF ===

=== RESULTS ===
Model       ACC    SEN    SPE    MCC    AUC
-------------------------------------------
rf       0.8235 0.8633 0.7712 0.6389 0.9164
Loaded pre-processed data: train=7834 val=980 test=980

=== Training SVM ===

=== RESULTS ===
Model       ACC    SEN    SPE    MCC    AUC
-------------------------------------------
svm      0.8255 0.8723 0.7642 0.6429 0.9036
Loaded pre-processed data: train=7834 val=980 test=980

=== Training XGB ===

=== RESULTS ===
Model       ACC    SEN    SPE    MCC    AUC
-------------------------------------------
xgb      0.8092 0.8417 0.7665 0.6103 0.8953


In [21]:
# Train GNN models (GCN, GAT, MPNN) - REQUIRES GPU
# ~5-10 minutes per model on GPU
# !python scripts/train.py --model gcn --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
# !python scripts/train.py --model gat --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
# !python scripts/train.py --model mpnn --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv

# With hyperparameter optimization (optional, slower):
!python scripts/train.py --model gcn --hpo --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
!python scripts/train.py --model gat --hpo --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
!python scripts/train.py --model mpnn --hpo --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
# Results saved to runs/<model>/best.pt
# Metrics saved to runs/results.json

Loaded pre-processed data: train=7834 val=980 test=980

=== Training GCN ===
[I 2026-08-23 11:33:52,370] A new study created in memory with name: no-name-174af684-91c1-47f8-9737-b08b90ea9617
[I 2026-08-23 11:34:01,422] Trial 0 finished with value: 0.6998205680738427 and parameters: {'hidden': 64, 'lr': 0.0015751320499779737, 'layers': 2}. Best is trial 0 with value: 0.6998205680738427.
[I 2026-08-23 11:34:10,518] Trial 1 finished with value: 0.6730203101669608 and parameters: {'hidden': 128, 'lr': 0.0015930522616241021, 'layers': 4}. Best is trial 0 with value: 0.6998205680738427.
[I 2026-08-23 11:34:17,108] Trial 2 finished with value: 0.699048544183521 and parameters: {'hidden': 64, 'lr': 0.00026587543983272726, 'layers': 2}. Best is trial 0 with value: 0.6998205680738427.
[I 2026-08-23 11:34:24,537] Trial 3 finished with value: 0.7026859644359983 and parameters: {'hidden': 128, 'lr': 0.0007309539835912913, 'layers': 2}. Best is trial 3 with value: 0.7026859644359983.
[I 2026-08-23 1

In [20]:
# OR train all models at once using pre-processed splits
!python scripts/train.py --model all --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv

# View results summary
import json
with open("runs/results.json") as f:
    results = json.load(f)

print(f"{'Model':<8} {'ACC':>6} {'SEN':>6} {'SPE':>6} {'MCC':>6} {'AUC':>6}")
print("-" * 44)
for name, m in results.items():
    auc_str = f"{m['auc']:.4f}" if m["auc"] is not None else "N/A"
    print(f"{name:<8} {m['acc']:.4f} {m['sen']:.4f} {m['spe']:.4f} {m['mcc']:.4f} {auc_str:>6}")

Loaded pre-processed data: train=7834 val=980 test=980

=== Training RF ===

=== Training SVM ===

=== Training XGB ===

=== Training GCN ===
  Epoch   1 | train_loss=0.6775 val_loss=0.6625 val_AUC=0.6523
  Epoch   2 | train_loss=0.6653 val_loss=0.6540 val_AUC=0.6655
  Epoch   3 | train_loss=0.6599 val_loss=0.6495 val_AUC=0.6730
  Epoch   4 | train_loss=0.6565 val_loss=0.6496 val_AUC=0.6767
  Epoch   5 | train_loss=0.6544 val_loss=0.6441 val_AUC=0.6795
  Epoch   6 | train_loss=0.6524 val_loss=0.6442 val_AUC=0.6837
  Epoch   7 | train_loss=0.6508 val_loss=0.6408 val_AUC=0.6844
  Epoch   8 | train_loss=0.6487 val_loss=0.6389 val_AUC=0.6842
  Epoch   9 | train_loss=0.6469 val_loss=0.6389 val_AUC=0.6856
  Epoch  10 | train_loss=0.6463 val_loss=0.6421 val_AUC=0.6839
  Epoch  11 | train_loss=0.6431 val_loss=0.6470 val_AUC=0.6840
  Epoch  12 | train_loss=0.6449 val_loss=0.6376 val_AUC=0.6851
  Epoch  13 | train_loss=0.6409 val_loss=0.6759 val_AUC=0.6804
  Epoch  14 | train_loss=0.6447 val_los

## 4. Screen New Compound Libraries

Use trained models to predict VEGFR2 activity on new SMILES.

In [22]:
# Prepare a screening library (example: create test CSV)
import pandas as pd

# Example: Your compound library
library_smiles = [
    "CC(=O)OC1=CC=CC=C1C(=O)O",  # Aspirin
    "CCO",                       # Ethanol
    "C1=CC=CC=C1",               # Benzene
    "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",  # Ibuprofen
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",   # Caffeine
    "CC(C)CC1=CC(=CC=C1)O",    # Naproxen-like
    "C[C@H](O)CC1=CC=CC=C1",   # Chiral molecule (R)
    "C[C@@H](O)CC1=CC=CC=C1",  # Chiral molecule (S)
    "C/C=C/C",                 # E-alkene
    "C/C=C\\C",                # Z-alkene
]

library_df = pd.DataFrame({"smiles": library_smiles})
library_df.to_csv("data/screen_library.csv", index=False)
print(f"Created library with {len(library_df)} compounds")
print(library_df)

Created library with 10 compounds
                          smiles
0       CC(=O)OC1=CC=CC=C1C(=O)O
1                            CCO
2                    C1=CC=CC=C1
3  CC(C)CC1=CC=C(C=C1)C(C)C(=O)O
4   CN1C=NC2=C1C(=O)N(C(=O)N2C)C
5           CC(C)CC1=CC(=CC=C1)O
6          C[C@H](O)CC1=CC=CC=C1
7         C[C@@H](O)CC1=CC=CC=C1
8                        C/C=C/C
9                        C/C=C\C


In [23]:
# Screen with a trained GNN model
# Change model_path to your best model
MODEL_PATH = "runs/gcn/best.pt"  # or gat/best.pt, mpnn/best.pt
INPUT_CSV = "data/screen_library.csv"
OUTPUT_CSV = "results/screen_gcn_results.csv"
THRESHOLD = 0.5

!python scripts/screen.py \
    --model {MODEL_PATH} \
    --input {INPUT_CSV} \
    --output {OUTPUT_CSV} \
    --threshold {THRESHOLD} \
    --batch-size 32

# View results
results_df = pd.read_csv(OUTPUT_CSV)
print(results_df[['smiles', 'probability', 'hit']].to_string(index=False))

Traceback (most recent call last):
  File "/content/ai-code/scripts/screen.py", line 111, in <module>
    sys.exit(main())
             ~~~~^^
  File "/content/ai-code/scripts/screen.py", line 93, in main
    out_df = screen_gnn(model_path, library_df, args.batch_size, device, args.threshold)
  File "/content/ai-code/scripts/screen.py", line 38, in screen_gnn
    batch = {k: v.to(device) for k, v in batch.items()}
                ^^^^
AttributeError: 'int' object has no attribute 'to'


FileNotFoundError: [Errno 2] No such file or directory: 'results/screen_gcn_results.csv'

In [24]:
# Screen with a trained ML model
MODEL_PATH = "runs/xgb/model.pkl"  # or rf/model.pkl, svm/model.pkl
INPUT_CSV = "data/screen_library.csv"
OUTPUT_CSV = "results/screen_xgb_results.csv"
THRESHOLD = 0.5

!python scripts/screen.py \
    --model {MODEL_PATH} \
    --input {INPUT_CSV} \
    --output {OUTPUT_CSV} \
    --threshold {THRESHOLD} \
    --batch-size 256

# View results
results_df = pd.read_csv(OUTPUT_CSV)
print(results_df[['smiles', 'probability', 'hit']].to_string(index=False))

Wrote 10 rows, 1 hits >= 0.5 to results/screen_xgb_results.csv
                       smiles  probability   hit
         CC(C)CC1=CC(=CC=C1)O     0.529493  True
 CN1C=NC2=C1C(=O)N(C(=O)N2C)C     0.343943 False
                          CCO     0.329788 False
                      C/C=C/C     0.306574 False
                      C/C=C\C     0.306574 False
                  C1=CC=CC=C1     0.270207 False
        C[C@H](O)CC1=CC=CC=C1     0.210912 False
       C[C@@H](O)CC1=CC=CC=C1     0.210912 False
CC(C)CC1=CC=C(C=C1)C(C)C(=O)O     0.110754 False
     CC(=O)OC1=CC=CC=C1C(=O)O     0.072210 False


## 5. Advanced: Custom Training & Screening

In [25]:
# Custom training with your own config
import yaml

custom_config = {
    "seed": 42,
    "paths": {
        "raw_csv": "data/raw/chembl_vegfr2.csv",
        "output_dir": "runs_custom"
    },
    "label": {"threshold_nM": 500},
    "split": {"test_size": 0.1, "val_frac_of_remaining": 0.111111},
    "fingerprint": {"radius": 2, "n_bits": 2048},
    "gnn": {
        "hidden": 128,
        "layers": 4,
        "heads": 8,
        "batch": 64,
        "lr": 0.0005,
        "epochs": 300,
        "patience": 20
    },
    "hpo": {"n_trials": 30}
}

with open("configs/custom_config.yaml", "w") as f:
    yaml.dump(custom_config, f)

print("Custom config saved. Train with:")
print("!python scripts/train.py --model gcn --config configs/custom_config.yaml")

Custom config saved. Train with:
!python scripts/train.py --model gcn --config configs/custom_config.yaml


In [26]:
# Custom screening with your own library CSV
# Your CSV must have a 'smiles' column

# Example: Load your library
# your_library = pd.read_csv("/content/drive/MyDrive/my_compounds.csv")
# your_library.to_csv("data/my_library.csv", index=False)

# Screen with multiple models and compare
import pandas as pd
import numpy as np

models_to_screen = [
    ("runs/gcn/best.pt", "GCN"),
    ("runs/gat/best.pt", "GAT"),
    ("runs/mpnn/best.pt", "MPNN"),
    ("runs/xgb/model.pkl", "XGBoost"),
    ("runs/rf/model.pkl", "RandomForest"),
]

library_df = pd.read_csv("data/screen_library.csv")
all_results = library_df[['smiles']].copy()

for model_path, model_name in models_to_screen:
    try:
        if model_path.endswith('.pt'):
            !python scripts/screen.py --model {model_path} --input data/screen_library.csv --output results/temp_{model_name}.csv --threshold 0.5 --batch-size 32
        else:
            !python scripts/screen.py --model {model_path} --input data/screen_library.csv --output results/temp_{model_name}.csv --threshold 0.5 --batch-size 256

        res = pd.read_csv(f"results/temp_{model_name}.csv")
        all_results[f"prob_{model_name}"] = res["probability"]
        all_results[f"hit_{model_name}"] = res["hit"]
        print(f"\u2705 {model_name} done")
    except Exception as e:
        print(f"\u274c {model_name} failed: {e}")

# Save combined results
all_results.to_csv("results/combined_screening.csv", index=False)
print("\nCombined results:")
print(all_results.to_string(index=False))

Traceback (most recent call last):
  File "/content/ai-code/scripts/screen.py", line 111, in <module>
    sys.exit(main())
             ~~~~^^
  File "/content/ai-code/scripts/screen.py", line 93, in main
    out_df = screen_gnn(model_path, library_df, args.batch_size, device, args.threshold)
  File "/content/ai-code/scripts/screen.py", line 38, in screen_gnn
    batch = {k: v.to(device) for k, v in batch.items()}
                ^^^^
AttributeError: 'int' object has no attribute 'to'
❌ GCN failed: [Errno 2] No such file or directory: 'results/temp_GCN.csv'
Traceback (most recent call last):
  File "/content/ai-code/scripts/screen.py", line 111, in <module>
    sys.exit(main())
             ~~~~^^
  File "/content/ai-code/scripts/screen.py", line 93, in main
    out_df = screen_gnn(model_path, library_df, args.batch_size, device, args.threshold)
  File "/content/ai-code/scripts/screen.py", line 38, in screen_gnn
    batch = {k: v.to(device) for k, v in batch.items()}
                ^^

In [27]:
# Ensemble prediction: average probabilities across models
ensemble_cols = [c for c in all_results.columns if c.startswith('prob_')]
all_results['prob_ensemble'] = all_results[ensemble_cols].mean(axis=1)
all_results['hit_ensemble'] = all_results['prob_ensemble'] >= 0.5

# Sort by ensemble probability
all_results = all_results.sort_values('prob_ensemble', ascending=False)

print("Top predictions (ensemble):")
display_cols = ['smiles', 'prob_ensemble', 'hit_ensemble'] + ensemble_cols
print(all_results[display_cols].to_string(index=False))

Top predictions (ensemble):
                       smiles  prob_ensemble  hit_ensemble  prob_XGBoost  prob_RandomForest
     CC(=O)OC1=CC=CC=C1C(=O)O       0.456413         False      0.529493           0.383333
                          CCO       0.348638         False      0.343943           0.353333
                  C1=CC=CC=C1       0.279894         False      0.329788           0.230000
CC(C)CC1=CC=C(C=C1)C(C)C(=O)O       0.243287         False      0.306574           0.180000
 CN1C=NC2=C1C(=O)N(C(=O)N2C)C       0.223287         False      0.306574           0.140000
         CC(C)CC1=CC(=CC=C1)O       0.205103         False      0.270207           0.140000
        C[C@H](O)CC1=CC=CC=C1       0.170456         False      0.210912           0.130000
       C[C@@H](O)CC1=CC=CC=C1       0.167123         False      0.210912           0.123333
                      C/C=C/C       0.117044         False      0.110754           0.123333
                      C/C=C\C       0.089438        

## 6. Download Results

In [28]:
# Download results to local machine
from google.colab import files

# Download combined screening results
files.download("results/combined_screening.csv")

# Download trained models (optional)
# files.download("runs/gcn/best.pt")
# files.download("runs/xgb/model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Tips & Troubleshooting

### GPU Issues
- **Runtime disconnected**: Colab free tier has limits. Use shorter epochs or smaller models.
- **OOM (Out of Memory)**: Reduce `batch` size in config.yaml (e.g., 64 → 32)
- **Slow training**: Ensure GPU is enabled (Runtime → Change runtime type → GPU)

### Data Issues
- **Download fails**: ChEMBL API may be slow. Retry or use a local fallback CSV.
- **Invalid SMILES**: Screen script skips invalid SMILES (shows NaN probability)

### Model Selection
- **Fast screening**: Use ML models (RF, XGBoost) - CPU only, very fast
- **Best accuracy**: GNN models (especially MPNN) - need GPU, capture 3D/steric effects
- **Chiral/E-Z sensitivity**: GNN models now use stereochemistry features (new in v2)

### Custom Library Format
Your screening CSV must have:
```csv
smiles,optional_id,optional_name
"CCO",CMPD001,Ethanol
"CC(=O)OC1=CC=CC=C1C(=O)O",CMPD002,Aspirin
```

### Threshold Tuning
- Default threshold: 0.5 (probability ≥ 0.5 = hit)
- For stricter hits: use 0.7-0.9
- For recall-focused: use 0.3-0.4
- Check `results.json` for model AUC to gauge reliability